# Aşama 14: TXS 0506+056 — önceden belirlenmiş zaman penceresi

Aşama 8'de 10 yılın tamamında patlama aradık ve "yüzlerce zaman penceresine baktık" cezası yedik (0.9σ). Oysa IceCube 2018'de bu patlamanın tarihlerini **yayımladı**. Bu aşamada sadece o pencereye bakıyoruz; pencere veriye bakmadan önce sabit olduğu için ceza yok.

IceCube'un 2018 makalesindeki iki pencere:
- **Kutu:** MJD 56937.81 – 57096.21 (Ekim 2014 – Şubat 2015, 158 gün)
- **Gauss:** merkez MJD 57004 (13 Aralık 2014), genişlik T_W = 110 gün (= 2σ, yani σ = 55 gün)

**Soru:** "IceCube'un 2014–15'te gördüğü patlama, yeniden işlenmiş 2021 verisinde hâlâ görünüyor mu?"

Model: Aşama 12'nin sezonlu, enerjili likelihood'u + zaman. Olay başına kanıt gücü artık üç parçalı: **yön × enerji × zaman**.

In [ ]:
import sys; sys.path.append('../src')
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import icecube_ps as ps
importlib.reload(ps)
assert float(ps.SURUM) >= float('14.0'), f"src/icecube_ps.py eski sürüm ({getattr(ps, 'SURUM', 'yok')}). Yenisini kopyala, kernel'i yeniden başlat."
try:
    from tqdm.auto import tqdm          # ilerleme çubuğu
except ImportError:
    tqdm = None

pd.set_option('display.float_format', '{:.3g}'.format)
DATA, RESULTS = Path('../data'), Path('../results')
ev = ps.veri_yukle(DATA)
B = ps.ArkaPlan(ev['dec'].values)
print('icecube_ps', ps.SURUM, '| olay:', f'{len(ev):,}')
from scipy import stats
sig0 = lambda p: max(0.0, ps.sigma(p))
yil = lambda mjd: 2000 + (np.asarray(mjd) - 51544.5) / 365.25

In [ ]:
M = ps.SezonluModel(ev, ps.etkin_alanlari_yukle(DATA), ps.canli_sureler_yukle(DATA), ps.smearingleri_yukle(DATA))
ra_k, dec_k = ps.KAYNAKLAR['TXS 0506+056']
pencereler = {
    'Kutu (MJD 56937.81–57096.21)': ps.kutu_pencere(56937.81, 57096.21),
    'Gauss (MJD 57004, σ = 55 gün)': ps.gauss_pencere(57004.0, 55.0),
}
modeller, fitler = {}, []
for ad, (pdf, cdf) in pencereler.items():
    Z = ps.ZamanPenceresiLikelihood(M, ra_k, dec_k, pdf, cdf)
    modeller[ad] = Z
    ns, gam, ts = Z.fit()
    fitler.append(dict(pencere=ad, ns=ns, gamma=gam, TS=ts))
pd.DataFrame(fitler).set_index('pencere')

## Anlamlılık

RA'ları karıştırıyoruz (olayların zamanlarına ve enerjilerine dokunmadan). ⏱️ Pencere başına 2000 deneme.

İki pencereyi birden test ettiğimiz için, IceCube'un yaptığı gibi en iyisini seçip **2 ile çarpıyoruz** (iki test → küçük bir deneme düzeltmesi).

In [ ]:
for f in fitler:
    Z = modeller[f['pencere']]
    dosya = RESULTS / f"asama14_arka_{'kutu' if 'Kutu' in f['pencere'] else 'gauss'}.npy"
    if dosya.exists():
        ts_a = np.load(dosya)
    else:
        ts_a = Z.arka_plan_denemeleri(2000, np.random.default_rng(1400), ilerleme=tqdm)
        np.save(dosya, ts_a)
    eta, sd = np.mean(ts_a > 0), np.mean(ts_a[ts_a > 0])
    p_mc = (np.sum(ts_a >= f['TS']) + 1) / (len(ts_a) + 1)
    p_k = eta * stats.chi2.sf(f['TS'], df=sd) if f['TS'] > 0 else 1.0
    f.update(p=p_k, sigma_MC=sig0(p_mc), sigma=sig0(p_k))

tablo14 = pd.DataFrame(fitler).set_index('pencere')
p_en = min(tablo14['p']) * 2
print(f"En iyi pencere, 2 test düzeltmesiyle: {sig0(p_en):.2f}σ")
tablo14.to_csv(RESULTS / 'asama14_txs_pencere.csv')
tablo14[['ns', 'gamma', 'TS', 'sigma_MC', 'sigma']]

In [ ]:
Z = modeller[list(modeller)[0]]
fig, ax = plt.subplots(figsize=(13, 4.5))
for s, d in Z.bant.items():
    psi = ps.acisal_uzaklik(d['ra'], d['dec'], ra_k, dec_k)
    y = psi < 1
    E = M.EP[s].oranlar_kaynak(dec_k, d['dec'][y], d['logE'][y])
    g_idx = int(np.argmin(np.abs(M.gamma - 2.0)))
    kanit = ps.sinyal_pdf(psi[y], d['sig'][y]) / d['B'][y] * E[g_idx]
    ax.scatter(yil(d['mjd'][y]), np.log10(kanit + 1e-3), s=12, alpha=0.7, color='#2e4057')
ax.axvspan(yil(56937.81), yil(57096.21), color='#d1495b', alpha=0.15, label='IceCube 2018 kutu penceresi')
t = np.linspace(56000, 58300, 2000)
ax.plot(yil(t), 6.3 * np.exp(-0.5 * ((t - 57004) / 55) ** 2), color='#d1495b', ls='--', lw=1.2, label='Gauss penceresi (şekil)')
ax.axvline(yil(58018.87), color='#edae49', lw=1.5, label='IceCube-170922A uyarısı (22 Eyl 2017)')
ax.set_xlim(2012.5, 2018.6)
ax.set_xlabel('yıl'); ax.set_ylabel('log₁₀(yön × enerji kanıtı, γ=2)')
ax.set_title('TXS 0506+056: 1° içindeki olaylar ve önceden belirlenmiş pencereler')
ax.legend(fontsize=9, loc='lower left'); ax.grid(alpha=0.3)
plt.savefig(RESULTS / 'asama14_txs_pencere.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
a8 = RESULTS / 'asama8_ts_arka.npy'
ozet = pd.DataFrame({
    'Bu aşama (kutu)': [f"{tablo14.iloc[0]['ns']:.1f}", f"{tablo14.iloc[0]['gamma']:.1f}", f"{tablo14.iloc[0]['sigma']:.2f}σ"],
    'Bu aşama (Gauss)': [f"{tablo14.iloc[1]['ns']:.1f}", f"{tablo14.iloc[1]['gamma']:.1f}", f"{tablo14.iloc[1]['sigma']:.2f}σ"],
    'IceCube 2018 (kutu)': ['13 ± 5 (fazlalık)', '2.2 ± 0.2', '3.5σ*'],
    'IceCube 2018 (Gauss)': ['—', '2.1 ± 0.2', '3.7σ*'],
}, index=['nₛ', 'γ', 'anlamlılık'])
ozet.to_csv(RESULTS / 'asama14_karsilastirma.csv')
ozet

\* IceCube'un değerleri 2008–2017 verisinin **eski işlenmiş** hâliyle ve zaman penceresi **taranarak** bulundu (o tarama için düzeltilmiş). Bizimki 2021'de yeniden işlenmiş veriyle ve **önceden sabit** pencereyle.

## Yorumlarken

- **Pencere sabit olduğu için bu sayı "temiz":** Aşama 8'deki gibi çok yere bakma cezası yok.
- **Anlamlılık 2018'deki değerden düşük çıkarsa** bu beklenen bir şey: IceCube'un kendi 2021 veri sürümü raporunda 2012–2015 olaylarının yeniden işlenmesinin TXS'nin anlamlılığını değiştirdiği yazıyor. Aynı soruya iki farklı veri sürümüyle bakıp farkı görmek, bilimde verinin nasıl işlendiğinin ne kadar önemli olduğunu gösteriyor.
- **γ ≈ 2 civarı çıkarsa** IceCube'un "sert spektrumlu patlama" bulgusuyla tutarlı.